In [1]:
import yfinance as yf
import pandas as pd

# ============================================
# QOIN — ICICI Bank Daily Data + RSI (From 2010)
# Ticker: ICICIBANK.NS (NSE)
# ============================================

In [2]:
df = yf.download("ICICIBANK.NS",start="2010-01-01",progress=False)
df.columns=df.columns.get_level_values(0)

In [3]:
data=pd.DataFrame()
data["Open"] =df["Open"]
data["High"] =df["High"]
data["Low"] =df["Low"]
data["Close"] =df["Close"]
data["Volume"] =df["Volume"]

# --- Calculate RSI (14-period) ---


In [4]:
period = 14
delta = data["Close"].diff()

In [5]:
gain=delta.where(delta>0,0)
loss = -delta.where(delta<0,0)

In [12]:
avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

In [13]:
rs=avg_gain/avg_loss
data["RSI_14"]=100-(100/(1+rs))

In [14]:
data=data.dropna()
latest=data.iloc[-1]

In [15]:
print(f"✅ ICICI Bank: {len(data)} trading days ({data.index[0].date()} to {data.index[-1].date()})")
print(f"   Latest Close: ₹{float(latest['Close']):.2f}")
print(f"   Latest RSI: {float(latest['RSI_14']):.2f}")
print(f"   Latest Volume: {int(latest['Volume']):,}")
 


✅ ICICI Bank: 4023 trading days (2010-01-21 to 2026-05-11)
   Latest Close: ₹1266.40
   Latest RSI: 43.23
   Latest Volume: 18,980,141


In [16]:
data.tail()

,Open,High,Low,Close,Volume,RSI_14
Date,,,,,,
2026-05-05,1264.000000,1266.699951,1245.500000,1251.300049,24447935,38.658206
2026-05-06,1262.000000,1282.800049,1252.099976,1279.500000,22909586,45.788327
2026-05-07,1284.699951,1294.000000,1273.199951,1279.000000,25028552,45.686927
2026-05-08,1279.599976,1279.599976,1261.199951,1264.800049,11137424,42.788811
2026-05-11,1259.400024,1275.699951,1252.000000,1266.400024,18980141,43.225816


In [17]:
data.to_csv("icici_bank_daily.csv")

1h Time frame

In [18]:
import yfinance as yf
import pandas as pd

# ============================================
# QOIN — ICICI Bank Intraday Data + RSI
# 1H: Max ~730 days back (yfinance limit)
# 4H: Aggregated from 1H data
# ============================================

TICKER = "ICICIBANK.NS"

def calculate_rsi(close, period=14):
    delta = close.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# -----------------------------------------------
# PART 1: Pull 1H data (max available)
# -----------------------------------------------
print("Pulling 1H data...")
df_1h = yf.download(TICKER, period="max", interval="1h", progress=False)
df_1h.columns = df_1h.columns.get_level_values(0)

data_1h = pd.DataFrame()
data_1h["Open"] = df_1h["Open"]
data_1h["High"] = df_1h["High"]
data_1h["Low"] = df_1h["Low"]
data_1h["Close"] = df_1h["Close"]
data_1h["Volume"] = df_1h["Volume"]
data_1h["RSI_14"] = calculate_rsi(data_1h["Close"])
data_1h = data_1h.dropna()

print(f"✅ 1H: {len(data_1h)} candles ({data_1h.index[0]} to {data_1h.index[-1]})")
print(f"   Latest Close: ₹{float(data_1h['Close'].iloc[-1]):.2f}")
print(f"   Latest RSI: {float(data_1h['RSI_14'].iloc[-1]):.2f}")

data_1h.to_csv("icici_1h.csv")
print(f"   💾 Saved: icici_1h.csv\n")

# -----------------------------------------------
# PART 2: Aggregate to 4H from 1H data
# -----------------------------------------------
print("Aggregating to 4H...")

data_4h = data_1h.resample("4h").agg({
    "Open": "first",
    "High": "max",
    "Low": "min",
    "Close": "last",
    "Volume": "sum"
}).dropna()

# Recalculate RSI on 4H candles
data_4h["RSI_14"] = calculate_rsi(data_4h["Close"])
data_4h = data_4h.dropna()

# Remove non-market hours (rows where Open/Close are same or volume is 0)
data_4h = data_4h[data_4h["Volume"] > 0]

print(f"✅ 4H: {len(data_4h)} candles ({data_4h.index[0]} to {data_4h.index[-1]})")
print(f"   Latest Close: ₹{float(data_4h['Close'].iloc[-1]):.2f}")
print(f"   Latest RSI: {float(data_4h['RSI_14'].iloc[-1]):.2f}")

data_4h.to_csv("icici_4h.csv")
print(f"   💾 Saved: icici_4h.csv\n")

# -----------------------------------------------
# Summary
# -----------------------------------------------
print("=" * 60)
print("Summary:")
print(f"  1H candles: {len(data_1h)} (from {data_1h.index[0].date()})")
print(f"  4H candles: {len(data_4h)} (from {data_4h.index[0].date()})")
print("=" * 60)
print("\nNOTE: yfinance only provides ~730 days of 1H data.")
print("For older intraday data, options are:")
print("  1. Zerodha Kite API (historical data endpoint)")
print("  2. TrueData / GlobalDataFeeds (paid, Indian market specialist)")
print("  3. Store data daily going forward to build your own database")

Pulling 1H data...
✅ 1H: 3408 candles (2024-05-14 09:45:00+00:00 to 2026-05-11 09:45:00+00:00)
   Latest Close: ₹1267.60
   Latest RSI: 47.98
   💾 Saved: icici_1h.csv

Aggregating to 4H...
✅ 4H: 1038 candles (2024-05-22 04:00:00+00:00 to 2026-05-11 08:00:00+00:00)
   Latest Close: ₹1267.60
   Latest RSI: 44.13
   💾 Saved: icici_4h.csv

Summary:
  1H candles: 3408 (from 2024-05-14)
  4H candles: 1038 (from 2024-05-22)

NOTE: yfinance only provides ~730 days of 1H data.
For older intraday data, options are:
  1. Zerodha Kite API (historical data endpoint)
  2. TrueData / GlobalDataFeeds (paid, Indian market specialist)
  3. Store data daily going forward to build your own database


## Bannk Nifty dta

In [19]:
import yfinance as yf
import pandas as pd

# ============================================
# QOIN — Bank Nifty Daily Data + RSI (From 2010)
# Ticker: ^NSEBANK (NSE Bank Nifty Index)
# ============================================

df = yf.download("^NSEBANK", start="2010-01-01", progress=False)
df.columns = df.columns.get_level_values(0)

data = pd.DataFrame()
data["Open"] = df["Open"]
data["High"] = df["High"]
data["Low"] = df["Low"]
data["Close"] = df["Close"]
data["Volume"] = df["Volume"]

# --- RSI with Wilder Smoothing (matches TradingView) ---
period = 14
delta = data["Close"].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
rs = avg_gain / avg_loss
data["RSI_14"] = 100 - (100 / (1 + rs))

data = data.dropna()
latest = data.iloc[-1]

print(f"✅ Bank Nifty: {len(data)} trading days ({data.index[0].date()} to {data.index[-1].date()})")
print(f"   Latest Close: {float(latest['Close']):.2f}")
print(f"   Latest RSI: {float(latest['RSI_14']):.2f}")
print(f"   Latest Volume: {int(latest['Volume']):,}")

data.to_csv("bank_nifty_daily.csv")
print(f"\n💾 Saved: bank_nifty_daily.csv")
print(f"\nLast 10 days:")
print(data.tail(10).round(2).to_string())

✅ Bank Nifty: 4016 trading days (2010-01-21 to 2026-05-11)
   Latest Close: 54439.90
   Latest RSI: 43.74
   Latest Volume: 0

💾 Saved: bank_nifty_daily.csv

Last 10 days:
                Open      High       Low     Close  Volume  RSI_14
Date                                                              
2026-04-27  56162.60  56474.95  55911.10  56264.30  235500   53.13
2026-04-28  55862.50  56138.05  55263.75  55400.35  292600   48.44
2026-04-29  55634.50  56178.75  55290.50  55403.60  313100   48.46
2026-04-30  54880.65  55111.60  54440.25  54863.35  299000   45.54
2026-05-04  54937.90  55602.30  54723.50  54878.50  282600   45.64
2026-05-05  54691.30  54888.55  54221.65  54547.05  379800   43.77
2026-05-06  55113.40  56078.80  54587.20  55981.05  653300   52.79
2026-05-07  56114.00  56334.15  55783.20  56047.40  463500   53.16
2026-05-08  55783.95  55797.70  55062.50  55310.55  429500   48.56
2026-05-11  54832.45  55002.45  54360.70  54439.90       0   43.74


## NIFTY 

In [20]:
import yfinance as yf
import pandas as pd

# ============================================
# QOIN — Nifty 50 Daily Data + RSI (From 2010)
# Ticker: ^NSEI (NSE Nifty 50 Index)
# ============================================

df = yf.download("^NSEI", start="2010-01-01", progress=False)
df.columns = df.columns.get_level_values(0)

data = pd.DataFrame()
data["Open"] = df["Open"]
data["High"] = df["High"]
data["Low"] = df["Low"]
data["Close"] = df["Close"]
data["Volume"] = df["Volume"]

# --- RSI with Wilder Smoothing (14-period, matches TradingView) ---
period = 14
delta = data["Close"].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
rs = avg_gain / avg_loss
data["RSI_14"] = 100 - (100 / (1 + rs))

data = data.dropna()
latest = data.iloc[-1]

print(f"✅ Nifty 50: {len(data)} trading days ({data.index[0].date()} to {data.index[-1].date()})")
print(f"   Latest Close: {float(latest['Close']):.2f}")
print(f"   Latest RSI: {float(latest['RSI_14']):.2f}")
print(f"   Latest Volume: {int(latest['Volume']):,}")

data.to_csv("nifty50_daily.csv")
print(f"\n💾 Saved: nifty50_daily.csv")
print(f"\nLast 10 days:")
print(data.tail(10).round(2).to_string())

✅ Nifty 50: 4001 trading days (2010-01-21 to 2026-05-11)
   Latest Close: 23815.85
   Latest RSI: 46.09
   Latest Volume: 0

💾 Saved: nifty50_daily.csv

Last 10 days:
                Open      High       Low     Close  Volume  RSI_14
Date                                                              
2026-04-27  23945.45  24130.70  23936.20  24092.70  430300   51.97
2026-04-28  24049.90  24181.80  23957.05  23995.70  554100   50.49
2026-04-29  24096.90  24334.70  24059.95  24177.65  531100   53.18
2026-04-30  23996.95  24087.45  23796.85  23997.55  505500   50.27
2026-05-04  24063.55  24290.20  24004.75  24119.30  419100   52.17
2026-05-05  24052.60  24081.70  23882.05  24032.80  363300   50.69
2026-05-06  24171.00  24356.50  23997.90  24330.95  429200   55.41
2026-05-07  24398.50  24482.10  24284.00  24326.65  440600   55.33
2026-05-08  24233.65  24253.80  24126.65  24176.15  335900   52.39
2026-05-11  23970.10  23997.45  23799.10  23815.85       0   46.09
